In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import os
import seaborn as sns
from scipy.stats import spearmanr

In [ ]:
DATASET = "ADNI"  # or "A4"

PROJECT_DIR = Path(os.environ["PROJECT_DIR"])
METADATA_FILE = Path(os.environ["METADATA_FILE"])
RESULTS_DIR = Path(os.environ["RESULTS_DIR"])

In [ ]:
inputs = ["anomaly", "tauSUVR_thresholded_tau_1_3"]
rois = ["Braak12", "GM_cortical", "Braak34", "Braak56"]

df_meta = pd.read_csv(METADATA_FILE)
df_meta.rename(columns={"identifier": "pid"}, inplace=True)

df_anomaly, df_tau = df_meta.copy(), df_meta.copy()
input_tau = Path(PROJECT_DIR, "outputs/roi_analysis_tauSUVR_thresholded_tau_1_3.csv")
input_anomaly = Path(RESULTS_DIR, "roi_analysis_anomaly.csv")
df_input_tau = pd.read_csv(input_tau)
df_input_anomaly = pd.read_csv(input_anomaly)

# Make sure no duplicate columns when merging
cols_to_use = df_input_tau.columns.difference(df_tau.columns)
df_tau = df_tau.merge(
    df_input_tau[["pid"] + list(cols_to_use)],
    on="pid",
    how="left"
)

cols_to_use = df_input_anomaly.columns.difference(df_anomaly.columns)
df_anomaly = df_anomaly.merge(
    df_input_anomaly[["pid"] + list(cols_to_use)],
    on="pid",
    how="left"
)

inputs_df = [df_anomaly, df_tau]

### Cohort values

In [ ]:
subject_col = "BIDS_ID"  # change if needed

print("\n===== ADNI SAMPLE COUNTS =====")

print("\nTOTAL")
print("Observations:", len(df_anomaly))
print("Participants:", df_anomaly[subject_col].nunique())

print("\nBY SPLIT")
for split, d in df_anomaly.groupby("split"):
    print(
        f"{split}: "
        f"{len(d)} observations, "
        f"{d[subject_col].nunique()} participants"
    )

print("\nBY SPLIT + DIAGNOSIS")
print(
    df_anomaly.groupby(["split", "dx"])
      .agg(
          observations=(subject_col, "size"),
          participants=(subject_col, "nunique")
      )
)

print("\nAMYLOID STATUS")
print(df_anomaly["amy"].value_counts(dropna=False))

print("\nTAU STATUS")
print(df_anomaly["tau"].value_counts(dropna=False))


print("\n===== NORMATIVE DEVELOPMENT COHORT =====")

normative = df_anomaly[
    (df_anomaly["dx"] == "CN") &
    (df_anomaly["amy"] == "amyNeg") &
    (df_anomaly["tau"] == "tauNeg")
]

print("Total normative observations:", len(normative))
print("Total normative participants:", normative[subject_col].nunique())

print("\nNormative by split:")
print(
    normative.groupby("split")
    .agg(
        observations=(subject_col, "size"),
        participants=(subject_col, "nunique")
    )
)

In [ ]:
print("\n===== A4 SAMPLE COUNTS =====")

print("Total observations:", len(df_anomaly))
print("Total participants:", df_anomaly[subject_col].nunique())

print("\nAmyloid status:")
print(df_anomaly["amy"].value_counts(dropna=False))

print("\nTau status:")
print(df_anomaly["tau"].value_counts(dropna=False))

In [ ]:
train = df_anomaly[df_anomaly["split"] == "train"].copy()

print("Training scans:", len(train))
print("Training participants:", train[subject_col].nunique())

print("\nAmyloid status:")
print(train["amy"].value_counts(dropna=False))

print("\nTau status:")
print(train["tau"].value_counts(dropna=False))

print("\nAmyloid × tau:")
print(pd.crosstab(train["amy"], train["tau"], margins=True))

print("\nAmyloid-positive training scans:")
cols = [subject_col, "amy", "tau", "dx"]
print(
    train.loc[train["amy"] == "amyPos", cols]
    .sort_values(subject_col)
    .to_string(index=False)
)

print(
    "\nUnique amyloid-positive training participants:",
    train.loc[train["amy"] == "amyPos", subject_col].nunique()
)

In [ ]:
# Define colors for metrics and biomarker groups
colors = {
    "Tau SUVr": "#4477AA",
    "DDPM": "#CC6677",
    "Z-score": "#F4A261",
    "Amyloid -": "#81B29A",
    "Amyloid +": "#117733",
    "Tau -": "#CC79A7",
    "Tau +": "#882255",
    "Anomaly": "#2A9D8F"
}

### Correlate with Cognitive Tests

In [ ]:
if DATASET == "ADNI":
    cog_tests = ["MMSE", "MOCA"]
elif DATASET == "A4":
    cog_tests = ["MMSE", "PACC_raw"]

fontsize = 29

reference_file = Path("ADNI_zscore_reference.csv")

# These MUST match the labels used when creating the boxplot reference file
metric_info = [
    (df_anomaly, "Mean anomaly\nscore", "anomaly_zscore"),
    (df_tau, "Mean tau\nSUVr > 1.3", "tau_zscore"),
]


def get_zscore_reference(df_current, metric_label, roi, dataset, reference_file):
    """
    ADNI:
        Calculate mean/std from the normative training split and save/update
        them in ADNI_zscore_reference.csv.

    A4:
        Load the corresponding ADNI training mean/std from the saved file.
    """

    if dataset == "ADNI":
        train_rows = df_current["split"] == "train"

        train_mean = df_current.loc[train_rows, roi].mean()
        train_std = df_current.loc[train_rows, roi].std()

        # Load existing reference table if available
        if reference_file.exists():
            reference_df = pd.read_csv(reference_file)
        else:
            reference_df = pd.DataFrame(
                columns=["Metric", "ROI", "mean", "std"]
            )

        # Remove an existing entry for this metric/ROI
        reference_df = reference_df[
            ~(
                (reference_df["Metric"] == metric_label) &
                (reference_df["ROI"] == roi)
            )
        ]

        # Add the current reference values
        new_row = pd.DataFrame([{
            "Metric": metric_label,
            "ROI": roi,
            "mean": train_mean,
            "std": train_std
        }])

        reference_df = pd.concat(
            [reference_df, new_row],
            ignore_index=True
        )

        reference_df.to_csv(reference_file, index=False)

    elif dataset == "A4":
        if not reference_file.exists():
            raise FileNotFoundError(
                f"{reference_file} does not exist. "
                "Run the ADNI analysis first to generate the normative "
                "z-score reference."
            )

        reference_df = pd.read_csv(reference_file)

        ref = reference_df[
            (reference_df["Metric"] == metric_label) &
            (reference_df["ROI"] == roi)
        ]

        if len(ref) != 1:
            raise ValueError(
                f"Could not find a unique ADNI z-score reference for "
                f"Metric='{metric_label}', ROI='{roi}'. "
                f"Found {len(ref)} rows."
            )

        train_mean = ref["mean"].iloc[0]
        train_std = ref["std"].iloc[0]

    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    if pd.isna(train_std) or train_std == 0:
        raise ValueError(
            f"Invalid standard deviation for {metric_label}, {roi}: "
            f"{train_std}"
        )

    return train_mean, train_std


for cog_test in cog_tests:
    for roi in rois:

        # ==============================================================
        # TAU SUVR
        # ==============================================================

        tau_mean, tau_std = get_zscore_reference(
            df_current=df_tau,
            metric_label="Mean tau\nSUVr > 1.3",
            roi=roi,
            dataset=DATASET,
            reference_file=reference_file
        )

        if DATASET == "ADNI":
            df_tau_eval = df_tau[
                df_tau["split"] == "test"
            ].copy()
        else:
            df_tau_eval = df_tau.copy()

        df_tau_eval = df_tau_eval.dropna(
            subset=[cog_test, roi]
        )

        # Z-score ONLY for visualisation
        df_tau_eval["tau_zscore"] = (
            (df_tau_eval[roi] - tau_mean) / tau_std
        )

        # Raw values for Spearman correlation
        roi_values_tau = df_tau_eval[roi].values
        cog_values_tau = df_tau_eval[cog_test].values

        corr_tau, pvalue_tau = spearmanr(
            roi_values_tau,
            cog_values_tau
        )

        # ==============================================================
        # ANOMALY SCORE
        # ==============================================================

        anomaly_mean, anomaly_std = get_zscore_reference(
            df_current=df_anomaly,
            metric_label="Mean anomaly\nscore",
            roi=roi,
            dataset=dataset,
            reference_file=reference_file
        )

        if dataset == "ADNI":
            df_anomaly_eval = df_anomaly[
                df_anomaly["split"] == "test"
            ].copy()
        else:
            df_anomaly_eval = df_anomaly.copy()

        df_anomaly_eval = df_anomaly_eval.dropna(
            subset=[cog_test, roi]
        )

        # Z-score ONLY for visualisation
        df_anomaly_eval["anomaly_zscore"] = (
            (df_anomaly_eval[roi] - anomaly_mean) / anomaly_std
        )

        # Raw values for Spearman correlation
        roi_values_anomaly = df_anomaly_eval[roi].values
        cog_values_anomaly = df_anomaly_eval[cog_test].values

        corr_anomaly, pvalue_anomaly = spearmanr(
            roi_values_anomaly,
            cog_values_anomaly
        )

        # ==============================================================
        # PLOTTING DATA
        # ==============================================================

        df_plot_tau = pd.DataFrame({
            "cog_test": cog_values_tau,
            "tau_zscore": df_tau_eval["tau_zscore"].values
        })

        df_plot_anomaly = pd.DataFrame({
            "cog_test": cog_values_anomaly,
            "anomaly_zscore": df_anomaly_eval["anomaly_zscore"].values
        })

        # ==============================================================
        # Plot
        # ==============================================================

        plt.rcParams.update({"font.size": fontsize})
        plt.figure(figsize=(8, 8), dpi=300)

        sns.regplot(
            x=df_plot_anomaly["cog_test"],
            y=df_plot_anomaly["anomaly_zscore"],
            line_kws={"color": colors["Anomaly"]},
            scatter_kws={"color": colors["Anomaly"]},
            ci=95,
            label=(
                f"r={corr_anomaly:.3f}, "
                f"{'p<0.001' if pvalue_anomaly < 0.001 else f'p={pvalue_anomaly:.3f}'}"
            )
        )

        ax = sns.regplot(
            x=df_plot_tau["cog_test"],
            y=df_plot_tau["tau_zscore"],
            line_kws={"color": colors["Tau SUVr"]},
            scatter_kws={"color": colors["Tau SUVr"]},
            ci=95,
            label=(
                f"r={corr_tau:.3f}, "
                f"{'p<0.001' if pvalue_tau < 0.001 else f'p={pvalue_tau:.3f}'}"
            )
        )

        # ==============================================================
        # Labels
        # ==============================================================

        if "_raw" in cog_test:
            xlabel = cog_test.replace("_raw", "")
        else:
            xlabel = cog_test

        plt.xlabel(xlabel)
        plt.ylabel("Z-scores")

        if "GM" in roi:
            plt.title("Cortical Grey Matter", fontsize=fontsize)
        elif "Braak12" in roi:
            plt.title("Braak I", fontsize=fontsize)
        elif "Braak34" in roi:
            plt.title("Braak III-IV", fontsize=fontsize)
        elif "Braak56" in roi:
            plt.title("Braak V-VI", fontsize=fontsize)

        plt.legend(fontsize=fontsize, loc="upper right")
        plt.tight_layout()
        plt.show()

#### ROC Analysis

In [ ]:
CN_EMCI = {
    "Cortical GM":{
        "Mean anomaly score": {
            "AUROC": 0.568,
            "CI": [0.50540163, 0.63107207]

        },
        "Mean tau SUVr > 1.3": {
            "AUROC": 0.53,
            "CI": [0.46533948, 0.59376244]
        }
    },
    "Braak12": {
        "Mean anomaly score": {
            "AUROC": 0.554,
            "CI": [0.48958571, 0.61816247]
        },
        "Mean tau SUVr > 1.3": {
            "AUROC": 0.538,
            "CI": [0.4721831, 0.60287744]
        }
    }
}

CN_Dem = {
    "Cortical GM":{
        "Mean anomaly score": {
            "AUROC": 0.841,
            "CI": [0.79832955, 0.88391443]
        },
        "Mean tau SUVr > 1.3": {
            "AUROC": 0.803,
            "CI": [0.75424097, 0.85093254]
        }
    },
    "Braak12": {
        "Mean anomaly score": {
            "AUROC": 0.841,
            "CI": [0.79504819, 0.88597161]
        },
        "Mean tau SUVr > 1.3": {
            "AUROC": 0.824,
            "CI": [0.77669443, 0.8705272 ]
        }
    }
}

# ROC

### ROC analysis - individual plots

In [ ]:
from math import tau
import numpy as np

def parse_ci(series):
    lower = []
    upper = []
    for ci in series:
        ci = str(ci).strip("[]")
        l, u = map(float, ci.split())
        lower.append(l)
        upper.append(u)
    return np.array(lower), np.array(upper)


stages_to_plot = {"CN vs MCI": "CN vs. MCI",
                  "CN vs Dementia": "CN vs. Dem",
                  "MCI vs Dementia": "MCI vs. Dem"}
rois_to_plot = {"GM_cortical": "Cortical\nGM",
                "Braak12": "Braak\nI"}
input_types_to_plot = ["anomaly", "tauSUVR_thresholded_tau_1_3"]

roc_csv = Path(RESULTS_DIR, "roc_analysis.csv")
df_roc = pd.read_csv(roc_csv)

fontsize = 36

for current_plot in stages_to_plot.keys():
    anomaly_rows = df_roc[
        (df_roc["ROI"].isin(rois_to_plot.keys())) &
        (df_roc["Classification"] == current_plot) &
        (df_roc["Input_Type"] == "anomaly")
    ].drop_duplicates(subset=["ROI"]).set_index("ROI").loc[rois_to_plot.keys()]

    tau_rows = df_roc[
        (df_roc["ROI"].isin(rois_to_plot.keys())) &
        (df_roc["Classification"] == current_plot) &
        (df_roc["Input_Type"] == "tauSUVR_thresholded_tau_1_3")
    ].drop_duplicates(subset=["ROI"]).set_index("ROI").loc[rois_to_plot.keys()]

    fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
    ax.tick_params(axis='both', labelsize=fontsize)


    bar_width = 0.3
    x = np.arange(len(rois_to_plot.keys()))  # the label locations

    anomaly_auroc = anomaly_rows["AUROC"].values
    tau_auroc = tau_rows["AUROC"].values

    anomaly_lower, anomaly_upper = parse_ci(anomaly_rows["95% AUC CI"])
    tau_lower, tau_upper = parse_ci(tau_rows["95% AUC CI"])
    anomaly_err = np.array([
        anomaly_auroc - anomaly_lower,  # lower
        anomaly_upper - anomaly_auroc   # upper
    ])

    tau_err = np.array([
        tau_auroc - tau_lower,
        tau_upper - tau_auroc
    ])

    # Plot bars with error bars
    rects1 = ax.bar(
        x - bar_width/2,
        anomaly_auroc,
        bar_width,
        yerr=anomaly_err,
        capsize=5,
        label="Mean anomaly score (ours)",
        color=colors["Anomaly"]
    )

    rects2 = ax.bar(
        x + bar_width/2,
        tau_auroc,
        bar_width,
        yerr=tau_err,
        capsize=5,
        label="Mean tau SUVr > 1.3 (baseline)",
        color=colors["Tau SUVr"]
    )

    for i in range(len(rois_to_plot.keys())):
        ax.text(x[i] - bar_width/2, anomaly_upper[i], f"{anomaly_auroc[i]:.3f}", ha='center', fontsize=fontsize, rotation=35)
        ax.text(x[i] + bar_width/2, tau_upper[i], f"{tau_auroc[i]:.3f}", ha='center', fontsize=fontsize, rotation=35)


    # Labels and formatting
    ax.set_ylabel('AUROC', fontsize=fontsize)
    ax.set_xlabel('ROI', fontsize=fontsize)
    ax.set_title(stages_to_plot[current_plot], fontsize=fontsize)
    ax.set_xticks(x)
    ax.set_xticklabels(rois_to_plot.values())
    ax.set_ylim(0, 1.05)  # AUROC ranges from 0 to 1

    plt.tight_layout()
    plt.show()


# Box Plots

In [ ]:
from scipy import stats

stratify = "tau"

if stratify == "amy":
    if DATASET == "ADNI":
        stratify_stages = ["amyNeg", "amyPos"]
    elif DATASET == "A4":
        stratify_stages = ["Ab.neg", "Ab.pos"]
    colors_boxplot = [colors["Amyloid -"], colors["Amyloid +"]]
elif stratify == "tau":
    stratify_stages = ["tauNeg", "tauPos"]
    colors_boxplot = [colors["Tau -"], colors["Tau +"]]

fontsize = 30
y_labels = ["Mean anomaly\nscore", "Mean tau\nSUVr > 1.3"]

# Loop over ROIs
for roi in rois:
    plot_data = []
    p_values = []

    # Build a combined DataFrame
    for df_current, y_label in zip(inputs_df, y_labels):
        # -----------------------------------
        # Z-score reference
        # ADNI: derive from normative training set and save
        # A4:   use saved ADNI normative reference
        # -----------------------------------

        reference_file = "ADNI_zscore_reference.csv"

        if DATASET == "ADNI":

            train_rows = df_current["split"] == "train"

            train_mean = df_current.loc[train_rows, roi].mean()
            train_std = df_current.loc[train_rows, roi].std()

            # Load existing reference file if it exists
            try:
                reference_df = pd.read_csv(reference_file)
            except FileNotFoundError:
                reference_df = pd.DataFrame(
                    columns=["Metric", "ROI", "mean", "std"]
                )

            # Remove old value for this metric/ROI, if present
            reference_df = reference_df[
                ~(
                    (reference_df["Metric"] == y_label) &
                    (reference_df["ROI"] == roi)
                )
            ]

            # Add current ADNI training reference
            new_row = pd.DataFrame([{
                "Metric": y_label,
                "ROI": roi,
                "mean": train_mean,
                "std": train_std
            }])

            reference_df = pd.concat(
                [reference_df, new_row],
                ignore_index=True
            )

            reference_df.to_csv(reference_file, index=False)


        elif DATASET == "A4":

            reference_df = pd.read_csv(reference_file)

            ref = reference_df[
                (reference_df["Metric"] == y_label) &
                (reference_df["ROI"] == roi)
            ]

            if len(ref) != 1:
                raise ValueError(
                    f"Could not find unique ADNI reference for "
                    f"{y_label}, {roi}"
                )

            train_mean = ref["mean"].iloc[0]
            train_std = ref["std"].iloc[0]

        # Avoid divide-by-zero
        if train_std == 0:
            print(f"Skipping {roi}, {y_label} due to zero std")
            continue

        # Copy dataframe and z-score normalize
        df_current = df_current.copy()

        df_current[f"{roi}_z"] = (
            df_current[roi] - train_mean
        ) / train_std
        if DATASET == "ADNI":
            df_current = df_current[df_current["split"] == "test"]

        # -----------------------------------
        # Build plotting dataframe
        # -----------------------------------
        for stage in stratify_stages:
            stage_rows = df_current.index[df_current[stratify] == stage]
            stage_scores = df_current.loc[stage_rows, f"{roi}_z"]
            for score in stage_scores:
                plot_data.append({
                    "Stratify Group": stage,
                    "Score": score,
                    "Metric": y_label
                })

    plot_df = pd.DataFrame(plot_data)

    # -----------------------------------
    # Group comparison: Mann-Whitney U
    # -----------------------------------
    for y_label in y_labels:

        group1 = plot_df.loc[
            (plot_df["Metric"] == y_label) &
            (plot_df["Stratify Group"] == stratify_stages[0]),
            "Score"
        ].dropna()

        group2 = plot_df.loc[
            (plot_df["Metric"] == y_label) &
            (plot_df["Stratify Group"] == stratify_stages[1]),
            "Score"
        ].dropna()

        u_stat, p_val = stats.mannwhitneyu(
            group1,
            group2,
            alternative="two-sided"
        )

        print(
            f"{roi}, {y_label}: "
            f"U = {u_stat:.1f}, "
            f"p = {p_val:.4e}, "
            f"n1 = {len(group1)}, "
            f"n2 = {len(group2)}, "
            f"median1 = {group1.median():.3f}, "
            f"median2 = {group2.median():.3f}"
        )

        p_values.append(p_val)

    # -----------------------------------
    # Plot
    # -----------------------------------
    plt.rcParams.update({'font.size': fontsize}) 
    plt.figure(figsize=(8, 8), dpi=300)
    ax = sns.boxplot(data=plot_df, x="Metric", y="Score", hue="Stratify Group", palette=colors_boxplot, width=0.5, showfliers=False)

    # -----------------------------------
    # Significance annotations
    # -----------------------------------

    # Get current y-limits after seaborn has drawn the boxplot
    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin

    # Put annotations above the visible boxplot
    annotation_y = ymax + 0.03 * yrange
    bar_height = 0.015 * yrange
    text_offset = 0.01 * yrange

    for i, (y_label, p_val) in enumerate(zip(y_labels, p_values)):

        # centres of the two hue boxes
        x1 = i - 0.2
        x2 = i + 0.2

        y = annotation_y

        ax.plot(
            [x1, x1, x2, x2],
            [y, y + bar_height, y + bar_height, y],
            lw=1.5,
            c="black",
            clip_on=False
        )

        if p_val < 0.001:
            star = "***"
        elif p_val < 0.01:
            star = "**"
        elif p_val < 0.05:
            star = "*"
        else:
            star = "ns"

        ax.text(
            i,
            y + bar_height + text_offset,
            star,
            ha="center",
            va="bottom",
            fontsize=18,
            color="black",
            clip_on=False
        )

    # Add enough headroom for bracket + stars
    ax.set_ylim(
        ymin,
        ymax + 0.12 * yrange
    )

    # -----------------------------------
    # Labels & formatting
    # -----------------------------------
    plt.ylabel("Z-scores", fontsize=fontsize)
    plt.xlabel("")
    if "GM" in roi:
        title = "Cortical Grey Matter"
    elif "Braak12" in roi:
        title = "Braak I"
    elif "Braak34" in roi:
        title = "Braak III-IV"
    elif "Braak56" in roi:
        title = "Braak V-VI"
    plt.title(title, fontsize=fontsize)
    ax.get_legend().remove()

    tick_positions = [-0.1, 1.1]  # Adjust to the number of metrics (here it's 2)
    print(tick_positions)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(y_labels)  # Set the correct labels for the x-axis
    plt.tight_layout()
    plt.show()


# Selective Interference Analysis

In [ ]:
csv_file = RESULTS_DIR / "selective_inference_reconstruction_error.csv"
df_plot = pd.read_csv(csv_file)
df_plot = df_plot.dropna(subset=["tau_global", "mse_difference"])

rho, p = spearmanr(
    df_plot["tau_global"],
    df_plot["mse_difference"]
)

print(f"Spearman rho = {rho:.3f}, p = {p:.3g}")

tau_labels = {
    False: "T-",
    True: "T+"
}

tau_colours = {
    False: "tab:blue",
    True: "tab:orange"
}

for status, group in df_plot.groupby("tau_status"):
    plt.scatter(
        group["tau_global"],
        group["mse_difference"],
        alpha=0.4,
        color=tau_colours[status],
        label=tau_labels[status]
    )

plt.axvline(1.3, linestyle="--", color="grey", alpha=0.7)

plt.xlabel("Global tau SUVR")
plt.ylabel("ΔMSE (selective − standard)")
plt.legend()
plt.show()


In [ ]:
import numpy as np

df_plot["group"] = np.where(
    (df_plot["DX"] == "CN") &
    (df_plot["amyloid_status"] == "Ab.neg") &
    (df_plot["tau_status"] == False),
    "CN/Aβ−/T−",
    "Other ADNI test"
)

print(df_plot["group"].value_counts())

summary = (
    df_plot.groupby("group")["mse_difference"]
      .agg(["count", "mean", "std", "median",
            lambda x: x.quantile(0.25),
            lambda x: x.quantile(0.75)])
)

summary.columns = [
    "n", "mean", "SD", "median", "Q1", "Q3"
]

print(summary)

In [ ]:
groups = ["CN/Aβ−/T−", "Other ADNI test"]

data = [
    df_plot.loc[df_plot["group"] == group, "mse_difference"].dropna()
    for group in groups
]

fig, ax = plt.subplots(figsize=(5, 5))

ax.boxplot(
    data,
    labels=groups,
    showfliers=False
)

rng = np.random.default_rng(42)

for i, values in enumerate(data, start=1):
    jitter = rng.normal(i, 0.04, size=len(values))
    ax.scatter(
        jitter,
        values,
        alpha=0.25,
        s=15
    )

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_ylabel("ΔMSE (selective − standard)")
ax.set_xlabel("")

plt.tight_layout()
plt.show()